In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [9]:
packages = [
    "io.delta:delta-spark_2.12:3.0.0",
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "com.amazonaws:aws-java-sdk-bundle:1.12.262"
]

In [10]:
spark = SparkSession.builder \
    .appName("silver_validation") \
    .master("local[*]") \
    .config("spark.jars.packages", ",".join(packages)) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [11]:
def silver_validation(tables: list) -> None:
    for table in tables:
        df = spark.read.format("delta").load(f"s3a://silver/{table}")
        print(f"Table {table}")
        print(f"Count: {df.count()}")
        df.printSchema()
        df.show(5, truncate=False)

In [12]:
dim_tables = ["branch", "currency", "channel", "merchant", "device", "person", "account", "loan_account"]
fact_tables = ["sign_in", "loan_payment", "exchange_rate", "transfer"]

In [13]:
silver_validation(dim_tables)

Table branch
Count: 34
root
 |-- id: integer (nullable = true)
 |-- branch_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- open_date: date (nullable = true)
 |-- cdc_operation: string (nullable = true)
 |-- cdc_timestamp: long (nullable = true)

+---+------------------+---------+----------+-------------+-------------+
|id |branch_name       |city     |open_date |cdc_operation|cdc_timestamp|
+---+------------------+---------+----------+-------------+-------------+
|1  |SafeBank Hai Phong|Hai Phong|2022-05-05|c            |1764223570643|
|3  |SafeBank Ca Mau   |Ca Mau   |2021-06-04|c            |1764223570646|
|7  |SafeBank Lam Dong |Lam Dong |2024-10-09|c            |1764223570647|
|8  |SafeBank An Giang |An Giang |2023-01-05|c            |1764223570647|
|11 |SafeBank Hung Yen |Hung Yen |2023-09-14|c            |1764223570649|
+---+------------------+---------+----------+-------------+-------------+
only showing top 5 rows

Table currency
Count: 12
root
 |-- code

In [14]:
silver_validation(fact_tables)

Table sign_in
Count: 5478
root
 |-- id: integer (nullable = true)
 |-- account_id: integer (nullable = true)
 |-- device_id: integer (nullable = true)
 |-- sign_in_time: timestamp (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- location_city: string (nullable = true)
 |-- status: string (nullable = true)
 |-- cdc_operation: string (nullable = true)
 |-- cdc_timestamp: long (nullable = true)

+---+----------+---------+--------------------------+---------------+--------------+-------+-------------+-------------+
|id |account_id|device_id|sign_in_time              |ip_address     |location_city |status |cdc_operation|cdc_timestamp|
+---+----------+---------+--------------------------+---------------+--------------+-------+-------------+-------------+
|2  |19        |15       |2025-11-27 13:06:12.514101|128.143.115.110|Sheppardbury  |FAILED |c            |1764223572857|
|4  |49        |16       |2025-11-27 13:06:17.575233|145.205.43.165 |South Linda   |SUCCESS|c           